In [2]:
import duckdb
import pandas as pd
import numpy as np

conn = duckdb.connect('../data/nyc_taxi.db')

df_raw = conn.execute("SELECT * FROM trips").df()
conn.close()

print(f"Wczytano {len(df_raw):,} rek.")

Wczytano 3,952,451 rek.


In [5]:
df_raw['tpep_pickup_datetime'] = pd.to_datetime(df_raw['tpep_pickup_datetime'])
df_raw['tpep_dropoff_datetime'] = pd.to_datetime(df_raw['tpep_dropoff_datetime'])

df_raw['duration_min'] = (df_raw['tpep_dropoff_datetime'] - df_raw['tpep_pickup_datetime']).dt.total_seconds() / 60.0

print("Max czas:", df_raw['duration_min'].max(), "min.")
print("Min czas:", df_raw['duration_min'].min(), "min.")

Max czas: 7620.6 min.
Min czas: -186.66666666666666 min.


In [11]:
print(f"{'Filtr':<25} {'Usuniętych':>12} {'%':>8}")
print("-" * 47)

checks = {
    'zakres_dat':   ~((df_raw['tpep_pickup_datetime'] >= '2026-03-01') &
                      (df_raw['tpep_pickup_datetime'] <  '2026-04-01')),
    'dystans':      ~((df_raw['trip_distance'] >= 0.05) &
                      (df_raw['trip_distance'] <= 50.0)),
    'fare_amount':  ~(df_raw['fare_amount'] >= 2.50),
    'total_amount': ~((df_raw['total_amount'] > 0.0) &
                      (df_raw['total_amount'] < 400.0)),
    'czas':         ~((df_raw['duration_min'] >= 1.0) &
                      (df_raw['duration_min'] <= 180.0)),
    'pasazerowie':  ~((df_raw['passenger_count'] >= 1) &
                      (df_raw['passenger_count'] <= 6)),
    'passenger == 0':  (df_raw['passenger_count'] == 0),
    'passenger NULL':  (df_raw['passenger_count'].isna()),
    'passenger > 6':   (df_raw['passenger_count'] > 6),
}

for name, mask in checks.items():
    n = mask.sum()
    print(f"{name:<25} {n:>12,} {n/len(df_raw)*100:>7.2f}%")

Filtr                       Usuniętych        %
-----------------------------------------------
zakres_dat                          19    0.00%
dystans                        147,658    3.74%
fare_amount                     24,497    0.62%
total_amount                    22,203    0.56%
czas                            90,371    2.29%
pasazerowie                    958,718   24.26%
passenger == 0                  12,967    0.33%
passenger NULL                 945,748   23.93%
passenger > 6                        3    0.00%


## Decyzja analityczna: passenger_count = 0

958,718 rekordów ma passenger_count = 0. 
Nie są to kursy bez pasażerów — terminal taksówkowy nie zarejestrował wartości.
Kurs fizycznie się odbył: dystans, czas, przychód są wiarygodne.

Stosujemy dwa zbiory:
- `df_clean` — wszystkie kursy (popyt, przychód, czas, dystans)
- `df_clean_passengers` — kursy z passenger_count >= 1 (analizy pasażerskie)

Ta sama filozofia obowiązuje dla payment_type = 0 (Unknown).

In [14]:
overlap = df_raw[(df_raw['passenger_count'].isna()) & (df_raw['payment_type'] == 0)]
print(f"Rekordy z NaN passenger_count ORAZ payment_type=0: {len(overlap):,}")

df_raw[df_raw['passenger_count'].isna()]['VendorID'].value_counts()

df_raw[df_raw['VendorID'] == 6][['VendorID', 'total_amount', 
    'trip_distance', 'tpep_pickup_datetime']].describe()

Rekordy z NaN passenger_count ORAZ payment_type=0: 945,748


,VendorID,total_amount,trip_distance,tpep_pickup_datetime
count,9456.0,9456.000000,9456.000000,9456
mean,6.0,33.093510,8.497859,2026-03-17 10:38:22.290186
min,6.0,0.000000,0.130000,2026-03-01 00:01:17
25%,6.0,21.597500,4.140000,2026-03-10 10:50:16.250000
50%,6.0,31.150000,7.730000,2026-03-17 20:38:41.500000
75%,6.0,42.372500,12.200000,2026-03-25 05:45:23.250000
max,6.0,145.290000,54.320000,2026-03-31 23:52:18
std,0.0,14.457753,5.242358,NaN


In [6]:
df_clean = df_raw[
    # Tylko marzec 2026 r. Pozostałe rekordy traktujemy jako wadliwe
    (df_raw['tpep_pickup_datetime'] >= '2026-03-01 00:00:00') & 
    (df_raw['tpep_pickup_datetime'] < '2026-04-01 00:00:00') &
    
    # Dystans: Min. 0.05 mil. Krótsze rekordy traktujemy jako błędy gps. Max. 50 mil. Dłuższe rekordy traktujemy jako wadliwe/podróże poza miejskie
    (df_raw['trip_distance'] >= 0.05) & 
    (df_raw['trip_distance'] <= 50.0) &
    
    # Min taryfa początkowa to $2.50/$3.00, odrzucamy ujemne i skrajnie wysokie wartości > $400
    (df_raw['fare_amount'] >= 2.50) &
    (df_raw['total_amount'] > 0.0) &
    (df_raw['total_amount'] < 400.0) &
    
    # Kurs musi trwać dłużej niż minutę. Maksymalny czas trwania kursu traktujemy jako 3 godziny.
    (df_raw['duration_min'] >= 1.0) & 
    (df_raw['duration_min'] <= 180.0) &
    
    # Rekordy związane z pasażerami będą akceptowane od 1 do 6 pasażerów.
    (df_raw['passenger_count'] >= 1) & 
    (df_raw['passenger_count'] <= 6)
].copy()

removed_count = len(df_raw) - len(df_clean)
removed_pct = (removed_count / len(df_raw)) * 100

print(f"Pozostało czystych rekordów: {len(df_clean):,}")
print(f"Usunięto {removed_count:,} ({removed_pct:.2f}%) wadliwych rekordów")

Pozostało czystych rekordów: 2,877,692
Usunięto 1,074,759 (27.19%) wadliwych rekordów
